# Exercise B1 — PINNs at Hand Scale: Forward and Inverse

**Companion to Lecture B** and to Supplement S1 of the lecture notes, *Worked PINN Examples
in One Dimension* (the six examples below follow that supplement). Benchmark problem throughout, from Lecture 2
(units $EA = L = n_0 = 1$): fixed–free bar, $u'' + 1 = 0$, $u(0)=0$, $u'(1)=0$,
exact solution $u_{\rm ex} = x - x^2/2$.

---
*Author: Ramses Sala · Copyright © 2025 Ramses Sala · Part of the course* Essential Concepts in FEM and Physics-Based ML. *Created with assistance of Claude Code for text, image and code refinement and generation.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# ---------- helpers provided for all exercises (read them once!) ----------

# num_grad is a transparent stand-in for automatic differentiation: real frameworks
# compute the same numbers exactly and much faster.
def num_grad(f, th, h=1e-6):
    """Numerical gradient of scalar f at parameter vector th (central differences)."""
    g = np.zeros_like(th)
    for i in range(len(th)):
        tp = th.copy(); tm = th.copy(); tp[i] += h; tm[i] -= h
        g[i] = (f(tp) - f(tm)) / (2*h)
    return g

def adam_minimize(f, th0, steps=2000, lr=0.02, verbose_every=None):
    """Adam: gradient descent with per-parameter step adaptation (industry standard)."""
    th = np.asarray(th0, float).copy()
    m = np.zeros_like(th); v = np.zeros_like(th)
    for k in range(steps):
        g = num_grad(f, th)
        m = 0.9*m + 0.1*g
        v = 0.999*v + 0.001*g*g
        mh = m/(1-0.9**(k+1)); vh = v/(1-0.999**(k+1))
        th -= lr*mh/(np.sqrt(vh)+1e-8)
        if verbose_every and k % verbose_every == 0:
            print(f"step {k:5d}   J = {f(th):.4e}")
    return th


In [ ]:
# ---------- a tiny tanh network, with exact derivatives (given) ----------
# g(x) = sum_j c_j * tanh(w_j x + b_j) + c0          parameters th = [c (H), w (H), b (H), c0]

def net(x, th, H):
    """Returns g(x), g'(x), g''(x) for the one-hidden-layer tanh network."""
    c = th[:H]; w = th[H:2*H]; b = th[2*H:3*H]; c0 = th[3*H]
    t  = np.tanh(np.outer(np.atleast_1d(x), w) + b)
    s2 = 1 - t**2                       # sech^2
    g   = t @ c + c0
    gp  = (s2 * w) @ c
    gpp = (-2 * t * s2 * w**2) @ c
    return g, gp, gpp

def u_hard(x, th, H):
    """u(x) = x * g(x): essential BC u(0)=0 built in (HARD). Returns u, u', u''."""
    x = np.atleast_1d(x).astype(float)
    g, gp, gpp = net(x, th, H)
    return x*g, g + x*gp, 2*gp + x*gpp

def init_net(H, seed=0):
    rng = np.random.default_rng(seed)
    return np.concatenate([rng.normal(0, 0.1, H), rng.normal(0, 1, H),
                           rng.normal(0, 1, H), [0.0]])


---
## Part 1 (Example 1) — The pencil PINN

Ansatz $\tilde u = a\,\varphi$, $\varphi = x - x^2/2$: **both** boundary conditions hard.

1. On paper: show $R(x) = 1 - a$ and hence $J(a) = (1-a)^2$ for any collocation points.
2. On paper: with $\eta = 1/4$, show the error $1 - a^k$ halves every step.
3. Confirm by code, and plot $\tilde u$ after 3 steps and after 10 steps against $u_{\rm ex}$.

In [ ]:
a = 0.0; eta = 0.25
history = [a]
for k in range(10):
    grad = ...          # TODO: J'(a)
    a = ...             # TODO: descent step
    history.append(a)
print("iterates:", [f"{h:.6f}" for h in history])
xx = np.linspace(0, 1, 200); phi = xx - xx**2/2
plt.plot(xx, phi, 'b', label='exact')
plt.plot(xx, history[3]*phi, 'g--', label='after 3 steps')
plt.plot(xx, history[10]*phi, 'r:', label='after 10 steps')
plt.legend(); plt.title('the pencil PINN'); plt.show()

In [ ]:
# ✅ check — Part 1
assert abs(history[5] - 31/32) < 1e-12, "After 5 steps: a = 31/32 exactly (powers of two!)."
assert abs(history[10] - (1 - 2.0**-10)) < 1e-12
print("OK. You have trained a PINN with a pencil (and verified it with silicon).")

---
## Part 2 (Example 2) — One soft boundary condition: the tilted valley

Ansatz $\tilde u = a x + b x^2$ (hard only at $x=0$); the free end becomes a penalty:
$$J(a,b) = (2b+1)^2 + \lambda\,(a + 2b)^2, \qquad \lambda = 1.$$

1. Implement the loss and run plain gradient descent ($\eta = 0.1$, from $(0,0)$, 300 steps),
   recording the path; plot it on the contour helper below.
2. **Experiments**: (i) at which $\eta$ does it diverge (theory: $\approx 0.117$)?
   (ii) set $\lambda = 10$ — does a stronger penalty help or hurt the walk?

In [ ]:
# given: contour plot helper
def plot_valley(path, lam=1.0):
    A, B = np.meshgrid(np.linspace(-0.4, 1.9, 200), np.linspace(-1.1, 0.4, 200))
    Jg = (2*B + 1)**2 + lam*(A + 2*B)**2
    plt.contour(A, B, Jg, levels=[0.02, 0.12, 0.45, 1.0, 2.0], colors='gray')
    p = np.array(path)
    plt.plot(p[:, 0], p[:, 1], 'r.-', ms=3, lw=0.8)
    plt.plot(1, -0.5, 'b*', ms=12, label='exact (1, -1/2)')
    plt.xlabel('a'); plt.ylabel('b'); plt.legend(); plt.show()

In [ ]:
lam = 1.0; eta = 0.1
p = np.array([0.0, 0.0]); path = [p.copy()]
def J2(p): a, b = p; return (2*b + 1)**2 + lam*(a + 2*b)**2
for k in range(300):
    g = ...            # TODO: gradient (use num_grad(J2, p), or derive it by hand)
    p = ...            # TODO: step
    path.append(p.copy())
print("final (a, b) =", p, "   J =", J2(p))
plot_valley(path, lam)

In [ ]:
# ✅ check — Part 2
assert np.allclose(p, [1.0, -0.5], atol=2e-2), "300 steps should land near (1, -1/2)."
print("OK. Note HOW it got there: fast across the valley, crawling along it.")

---
## Part 3 (Example 3) — A real forward PINN

Now the tanh network (helpers above): $\tilde u = x\cdot g(x)$, hard at $x=0$;
loss = mean squared residual $(\tilde u'' + 1)^2$ at 50 collocation points
$+$ penalty $\lambda\,(\tilde u'(1))^2$ with $\lambda = 1$ for the free end. Train with Adam, then compare with
$u_{\rm ex}$ — and with the two-element FEM solution of Lecture 2, which is exact at the
nodes after one $2\times2$ solve. Feel the asymmetry.

In [ ]:
H = 6
x_col = np.linspace(0.01, 1.0, 50)

def pinn_loss(th):
    u, du, ddu = u_hard(x_col, th, H)
    residual = ...                     # TODO: the ODE residual at the collocation points
    u1, du1, _ = u_hard(np.array([1.0]), th, H)
    return ...                         # TODO: mean residual^2 + penalty (weight 1) on du1[0]

th = adam_minimize(pinn_loss, init_net(H, seed=0), steps=2000, lr=0.02, verbose_every=500)
xx = np.linspace(0, 1, 300)
u_fit, _, _ = u_hard(xx, th, H)
max_err = np.max(np.abs(u_fit - (xx - xx**2/2)))
print(f"max |u_PINN - u_exact| = {max_err:.2e}")
plt.plot(xx, xx - xx**2/2, 'b', label='exact')
plt.plot(xx, u_fit, 'r--', label='PINN')
plt.plot([0, .5, 1], [0, .375, .5], 'ks', label='FEM, 2 elements (exact at nodes)')
plt.legend(); plt.show()

In [ ]:
# ✅ check — Part 3
assert max_err < 1e-2, "2000 Adam steps should bring the max error below 1e-2."
print(f"OK (max error {max_err:.1e}). Two thousand steps vs one 2x2 solve: remember the scorecard.")

---
## Part 4 (Example 4) — Inverse warm-up: interrogating the stepped bar

The stepped steel bar of Lecture 1 (Section 1.7): known tip load $F = 10$ kN, measured
$u_2 = 0.25$ mm, $u_3 = 0.50$ mm. The stiffnesses $k_1, k_2$ are unknown.

1. **On paper**: read the two node equilibrium equations backwards and solve for $k_1, k_2$.
2. Confirm in code (SI units). *Identification is equilibrium read in reverse.*

In [ ]:
F  = 10e3          # N
u2 = 0.25e-3       # m
u3 = 0.50e-3       # m
k2 = ...           # TODO: from node-3 equilibrium
k1 = ...           # TODO: from node-2 equilibrium
print(f"k1 = {k1/1e6:.1f} MN/m   k2 = {k2/1e6:.1f} MN/m")

In [ ]:
# ✅ check — Part 4
assert np.isclose(k1, 40e6) and np.isclose(k2, 40e6), "Both should come out as 40 MN/m (Lecture 1!)."
print("OK — you recovered exactly the stiffnesses of the Lecture 1 mini-example.")

---
## Part 5 (Example 5) — The smallest inverse PINN

Bar fixed at 0, tip force $\bar F = 2$, no distributed load. **Unknown**: $\kappa = EA$.
**Data**: one measurement $u^{\rm meas} = 0.5$ at $x_d = 1$. Ansatz $\tilde u = w x$.

$$J(w, \kappa) = (\kappa w - \bar F)^2 + (w\,x_d - u^{\rm meas})^2$$

1. **On paper**: minimize by inspection ($J=0$ is achievable). Expect $w = 0.5$, $\kappa = 4$.
2. Minimize numerically with Adam from $(w, \kappa) = (1, 1)$.
3. **Experiment**: delete the data term and re-run several times. What happens, and why?
   (Hint: which *combinations* of $(w,\kappa)$ does physics alone constrain?)

In [ ]:
F_bar, x_d, u_meas = 2.0, 1.0, 0.5
def J_inv(th):
    w, kappa = th
    return ...          # TODO: physics term + data term

th = adam_minimize(J_inv, np.array([1.0, 1.0]), steps=3000, lr=0.02, verbose_every=1000)
w, kappa = th
print(f"w = {w:.4f}   kappa = {kappa:.4f}   (expected 0.5 and 4)")

In [ ]:
# ✅ check — Part 5
assert abs(w - 0.5) < 0.01 and abs(kappa - 4.0) < 0.05, "Expect w = 0.5, kappa = 4."
print("OK. Data pinned the field; physics then revealed the parameter.")

---
## Part 6 (Example 6) — Finale: damage detection

A bar with a hidden weakened zone:
$$EA(x;\,d,\,x_c) = 1 - d\,e^{-\left((x-x_c)/w_z\right)^2}, \qquad w_z = 0.1 \text{ (known)},$$
tip force $\bar F = 1$, fixed at $0$. Measurements of $u$ at 8 stations (generated below from the
true damage $d = 0.5$, $x_c = 0.6$ — which your optimizer is of course not told).

1. **On paper**: integrate the balance once to justify the first-order residual
   $R(x) = EA(x)\,\tilde u'(x) - \bar F$ (fewer derivatives — the weak-form gift again).
2. Build the joint loss over network weights **and** $(d, x_c)$:
   $$J = \tfrac1m\textstyle\sum_i R(x_i)^2 + 20 \cdot \text{mean}\bigl(\tilde u(x_j^{\rm meas}) - u_j^{\rm meas}\bigr)^2$$
3. Train (~3000 Adam steps) and report the identified $(d, x_c)$. Plot true vs identified $EA(x)$.
4. **Bonus**: add noise `0.002*rng.standard_normal(8)` to the data. How robust is the identification?

In [ ]:
# given: the stiffness profile, synthetic measurements (via quadrature), and setup
F_bar, w_z = 1.0, 0.1
d_true, xc_true = 0.5, 0.6
EA = lambda x, d, xc: 1.0 - d*np.exp(-((x - xc)/w_z)**2)

x_fine = np.linspace(0, 1, 2001)
u_fine = np.concatenate([[0], np.cumsum(F_bar/EA(x_fine, d_true, xc_true))[:-1]*(x_fine[1]-x_fine[0])])
x_meas = np.linspace(0.1, 1.0, 8)
u_meas = np.interp(x_meas, x_fine, u_fine)          # add noise here for the bonus task

H = 6
x_col = np.linspace(0.01, 1.0, 50)
# parameter vector: [network params (3H+1) | d | xc]
rng = np.random.default_rng(0)
th0 = np.concatenate([rng.normal(0, 0.1, H), rng.normal(0, 1, H),
                      rng.normal(0, 1, H), [1.0], [0.1], [0.4]])

In [ ]:
def damage_loss(th):
    th_net, d, xc = th[:3*H+1], th[-2], th[-1]
    u, du, _ = u_hard(x_col, th_net, H)
    R = ...                    # TODO: first-order residual  EA(x; d, xc) * u' - F_bar
    um, _, _ = u_hard(x_meas, th_net, H)
    return ...                 # TODO: mean R^2  +  20 * mean (data misfit)^2

th = adam_minimize(damage_loss, th0, steps=3000, lr=0.02, verbose_every=600)
d_id, xc_id = th[-2], th[-1]
print(f"after 3000 steps: J = {damage_loss(th):.1e}")
print(f"identified: d = {d_id:.4f} (true {d_true})   xc = {xc_id:.4f} (true {xc_true})")
plt.plot(x_fine, EA(x_fine, d_true, xc_true), 'b', label='true EA(x)')
plt.plot(x_fine, EA(x_fine, d_id, xc_id), 'r--', label='identified EA(x)')
plt.plot(x_meas, u_meas*0 + 1.05, 'gv', label='measurement stations')
plt.legend(); plt.xlabel('x'); plt.title('damage detection'); plt.show()

In [ ]:
# ✅ check — Part 6
assert abs(d_id - d_true) < 0.05, "Damage depth should be identified within 0.05."
assert abs(xc_id - xc_true) < 0.03, "Damage location should be identified within 0.03."
print("OK. Eight measured numbers + one differential equation = a located, quantified flaw.")

---
## Closing reflections (markdown answers)

1. In Part 6, nothing stops you from exploiting static determinacy: $u'(x) = \bar F/EA(x)$,
   so $(d, x_c)$ could be fitted directly without any network. Do it if you like — then explain
   in two sentences why this shortcut dies in 2D (hint: redundancy), which is where the joint
   training you just practiced becomes the actual method.
2. Which boundary conditions in this notebook were *hard* and which *soft*? Reclassify every part.
3. Transpose Part 6 to the thermal rod of Exercise 2: what plays $EA$, $\bar F$, $u^{\rm meas}$?
   What physical scenario does 'damage' become?